https://www.canva.com/design/DAHD7vQGNnY/hV7TnxcM6p2_rzNTr12GSA/edit

https://techdocs.gbif.org/en/openapi/

In [1]:
import os
from dotenv import load_dotenv, dotenv_values 

In [2]:
import time
import zipfile
import io
import pandas as pd
from pygbif import occurrences, species

In [3]:
GBIF_user  = os.getenv("GBIF_user")
GBIF_pwd   = os.getenv("GBIF_pwd")
GBIF_email = os.getenv("GBIF_email")

In [4]:
data_folder = "data_csv"
data_zip_folder = "data_zip"

In [ ]:
TAXA = {
    # Honigbienen
    "Apis mellifera":       species.name_backbone(scientificName="Apis mellifera"),
    # Hummeln (Gattung)
    "Bombus":               species.name_backbone(scientificName="Bombus"),
    # Gemeine Wespe & Deutsche Wespe (Gattung Vespula)
    "Vespula":              species.name_backbone(scientificName="Vespula"),
    # Langkopfwespen (Dolichovespula – Hornissen-ähnliche)
    "Dolichovespula":       species.name_backbone(scientificName="Dolichovespula"),
    # Echte Hornisse
    "Vespa crabro":         species.name_backbone(scientificName="Vespa crabro"),
    # Asiatische Hornisse (invasiv)
    "Vespa velutina":       species.name_backbone(scientificName="Vespa velutina"),
}

for name, key in TAXA.items():
    print(f"  {name:30s} → taxonKey={key}")

  Apis mellifera                 → taxonKey={'usage': {'key': '1341976', 'name': 'Apis mellifera Linnaeus, 1758', 'canonicalName': 'Apis mellifera', 'authorship': 'Linnaeus, 1758', 'rank': 'SPECIES', 'code': 'ZOOLOGICAL', 'status': 'ACCEPTED', 'genericName': 'Apis', 'specificEpithet': 'mellifera', 'type': 'SCIENTIFIC', 'formattedName': '<i>Apis</i> <i>mellifera</i> Linnaeus, 1758'}, 'classification': [{'key': '1', 'name': 'Animalia', 'rank': 'KINGDOM'}, {'key': '54', 'name': 'Arthropoda', 'rank': 'PHYLUM'}, {'key': '216', 'name': 'Insecta', 'rank': 'CLASS'}, {'key': '1457', 'name': 'Hymenoptera', 'rank': 'ORDER'}, {'key': '4334', 'name': 'Apidae', 'rank': 'FAMILY'}, {'key': '1334757', 'name': 'Apis', 'rank': 'GENUS'}, {'key': '1341976', 'name': 'Apis mellifera', 'rank': 'SPECIES'}], 'diagnostics': {'matchType': 'EXACT', 'confidence': 99, 'timeTaken': 11, 'timings': {'nameNRank': 0, 'sciNameMatch': 12, 'nameParse': 1, 'luceneMatch': 11}}, 'synonym': False}
  Bombus                      

In [6]:
taxa_keys_dict = {}
for name, key in TAXA.items():
    taxa_keys_dict[name] = key['usage']['key']

taxa_keys_dict

{'Apis mellifera': '1341976',
 'Bombus': '1340278',
 'Vespula': '1311631',
 'Dolichovespula': '1311794',
 'Vespa crabro': '1311527',
 'Vespa velutina': '1311477'}

In [7]:
name_tmp = 'Vespa velutina'

In [8]:
key_tmp = taxa_keys_dict[name_tmp]
key_tmp

'1311477'

In [ ]:
# 1. Download 
res = occurrences.download(
    [f"taxonKey = {key_tmp}", "country = DE"],
    user  = GBIF_user,
    pwd   = GBIF_pwd,
    email = GBIF_email
)

download_key = res[0]  # z.B. "0001234-231120084113126"
print(f"Download-Key: {download_key}")

# 2. Wait for GBIF
print("Wait for GBIF...")
while True:
    meta = occurrences.download_meta(download_key)
    status = meta["status"]
    print(f"  Status: {status}", end="\r")
    if status == "SUCCEEDED":
        print("Meta Data done!")
        break
    elif status == "FAILED":
        raise RuntimeError("GBIF Download failed")
    time.sleep(30)  # alle 30 Sekunden prüfen

INFO:Your download key is 0012426-260507073636908


Download-Key: 0012426-260507073636908
Wait for GBIF...
\Meta Data done!DED


In [10]:
# 3. ZIP download
occurrences.download_get(download_key, path=f"{data_zip_folder}") # → saves e.g. "0001234-231120084113126.zip"

INFO:Download file size: 1106919 bytes
INFO:On disk at data_zip/0012426-260507073636908.zip


{'path': 'data_zip/0012426-260507073636908.zip',
 'size': 1106919,
 'key': '0012426-260507073636908'}

In [ ]:
# 4. ZIP to DataFrme
zip_path = f"{data_zip_folder}/{download_key}.zip"

with zipfile.ZipFile(zip_path) as z:
    name = [n for n in z.namelist()][0] 
    print(f"Reading: {name}")
    with z.open(name) as f:
        df = pd.read_csv(f, sep="\t", low_memory=False)

print(f"\nDataframe loaded: {len(df):,} Rows, {len(df.columns)} Columns")

  Reading: 0012426-260507073636908.csv

Dataframe loaded: 18,519 Rows, 50 Columns


In [12]:
df.columns

Index(['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum', 'class',
       'order', 'family', 'genus', 'species', 'infraspecificEpithet',
       'taxonRank', 'scientificName', 'verbatimScientificName',
       'verbatimScientificNameAuthorship', 'countryCode', 'locality',
       'stateProvince', 'occurrenceStatus', 'individualCount',
       'publishingOrgKey', 'decimalLatitude', 'decimalLongitude',
       'coordinateUncertaintyInMeters', 'coordinatePrecision', 'elevation',
       'elevationAccuracy', 'depth', 'depthAccuracy', 'eventDate', 'day',
       'month', 'year', 'taxonKey', 'speciesKey', 'basisOfRecord',
       'institutionCode', 'collectionCode', 'catalogNumber', 'recordNumber',
       'identifiedBy', 'dateIdentified', 'license', 'rightsHolder',
       'recordedBy', 'typeStatus', 'establishmentMeans', 'lastInterpreted',
       'mediaType', 'issue'],
      dtype='object')

In [13]:
df.head(n=3)

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
0,6253332351,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/35767...,Animalia,Arthropoda,Insecta,Hymenoptera,Vespidae,Vespa,Vespa velutina,...,thomas_lange,2026-05-02T21:26:53,CC_BY_NC_4_0,Anton Dangl,Anton Dangl,NaN,NaN,2026-05-08T22:19:55.562Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
1,6253321064,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/35595...,Animalia,Arthropoda,Insecta,Hymenoptera,Vespidae,Vespa,Vespa velutina,...,woodenfox,2026-04-28T05:38:57,CC_BY_NC_4_0,woodenfox,woodenfox,NaN,NaN,2026-05-08T22:25:33.914Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...
2,6253085198,50c9509d-22c7-4a22-a47d-8c48425ef4a7,https://www.inaturalist.org/observations/35681...,Animalia,Arthropoda,Insecta,Hymenoptera,Vespidae,Vespa,Vespa velutina,...,Wolf-Achim and Hanna Roland,2026-04-30T11:15:30,CC_BY_NC_4_0,Wolf-Achim and Hanna Roland,Wolf-Achim and Hanna Roland,NaN,NaN,2026-05-08T22:13:17.987Z,StillImage,COORDINATE_ROUNDED;CONTINENT_DERIVED_FROM_COOR...


In [14]:
dates_strings = df.eventDate.unique().tolist()
dates_strings = [str(date)[:4] for date in dates_strings]
dates_years = list(set(dates_strings))
dates_years = [int(date) for date in dates_years if date.isdigit()]
sorted(dates_years)

[1990, 2014, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [15]:
df.to_csv(f"{data_folder}/{name_tmp}.csv", index=False)